In [ ]:
AUTHOR_NAME = "Dr. Nidhika Chauhan"

AUTHOR_EMAIL = "nidhi29.chauhan@gmail.com"

import json

import logging

import os

from pathlib import Path

import numpy as np

import pandas as pd

import xgboost as xgb

import matplotlib.pyplot as plt

from scipy import stats

from sklearn.isotonic import IsotonicRegression

N_DEVICES = 1000

SIMULATION_DAYS = 60

SIMULATION_HOURS = SIMULATION_DAYS * 24

TRAIN_FRACTION, CALIBRATION_FRACTION = (0.55, 0.2)

EMERGENCY_MISS_TARGET = 0.1

CAUSE_NAMES = [
    "Dead_Battery",
    "Gateway_Outage",
    "OS_App_Kill",
    "Bluetooth_Drop",
    "Patient_NonWear",
    "Hardware_Fault",
    "Clinical_Emergency",
]

N_CAUSES = len(CAUSE_NAMES)

(
    CAUSE_DEAD_BATTERY,
    CAUSE_GATEWAY_OUTAGE,
    CAUSE_APP_TERMINATION,
    CAUSE_BLUETOOTH_DROP,
    CAUSE_NONWEAR,
    CAUSE_HARDWARE_FAULT,
    CAUSE_CLINICAL_EMERGENCY,
) = range(N_CAUSES)

ACTION_WAIT, ACTION_NUDGE, ACTION_RECONNECT, ACTION_NURSE, ACTION_REPLACE = range(5)

DEFAULT_LOSS_PARAMETERS = dict(
    cost=np.array([0, 2, 1, 75, 250], float),
    lat_nudge=1.0,
    lat_recon=0.25,
    lat_nurse=1.0,
    lat_replace=48.0,
    ver_nudge=2.0,
    ver_recon=0.5,
    ver_replace=48.0,
    p_nudge=np.array([0.8, 0.0, 0.05, 0.1, 0.6, 0.0, 0.0]),
    p_recon=np.array([0.0, 0.02, 0.85, 0.9, 0.02, 0.05, 0.0]),
    p_repl=np.array([0.0, 0.0, 0.9, 0.9, 0.0, 1.0, 0.0]),
    p_hw_needs_replace=0.7,
    lam_data=4.0,
    lam_emerg=200.0,
    wait_T=12.0,
)

CAUSE_TO_GREEDY_ACTION = np.array(
    [
        ACTION_NUDGE,
        ACTION_WAIT,
        ACTION_RECONNECT,
        ACTION_RECONNECT,
        ACTION_NUDGE,
        ACTION_REPLACE,
        ACTION_NURSE,
    ]
)

FEATURE_COLUMNS = [
    "batt_level",
    "batt_slope",
    "hrs_since_charge",
    "motion_E",
    "accel_peak",
    "contact_q",
    "rssi_trend",
    "vitals_anom",
    "cohort_gw",
    "cohort_os",
    "cohort_model",
    "cohort_gw_3h",
    "cohort_os_3h",
    "cohort_model_3h",
    "prior_eps",
    "time_of_day",
    "gateway_id",
    "os_version",
    "phone_model",
]

ABLATION_FEATURE_GROUPS = {
    "no_cohort": [
        "cohort_gw",
        "cohort_os",
        "cohort_model",
        "cohort_gw_3h",
        "cohort_os_3h",
        "cohort_model_3h",
    ],
    "no_battery": ["batt_level", "batt_slope", "hrs_since_charge"],
    "no_motion": ["motion_E", "accel_peak", "contact_q"],
    "no_vitals": ["vitals_anom"],
    "no_history": ["prior_eps"],
}

PROBABILITY_CALIBRATION_FRACTION = 0.1

POLICY_TUNING_FRACTION = 0.1

CALIBRATION_FRACTION = PROBABILITY_CALIBRATION_FRACTION + POLICY_TUNING_FRACTION

MODEL_THREADS = -1

MOTION_SAMPLE_RATE_HZ = 100

MOTION_DOWNSAMPLE_FACTOR = 10

MOTION_INPUT_SCALE_TO_G = 1.0

OBSERVATION_DELAY_H = 1.0

RULE_VITALS_THRESHOLDS = [-float("inf"), 1.0, 1.5, 2.0, 2.5, 3.0, float("inf")]

RULE_ACCELERATION_THRESHOLDS = [0.2, 0.35, 0.45, 0.6, 0.8, float("inf")]

INFRASTRUCTURE_TICKET_COST = 5.0

INFRASTRUCTURE_REPAIR_COST = 15.0

INFRASTRUCTURE_REPAIR_DELAY_H = 1.0

CLASSIFIER_PARAMETERS = dict(
    n_estimators=150, max_depth=4, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8
)

CLASS_WEIGHT_EXPONENT = 0.5

RESULTS_DIRECTORY = os.environ.get("WBAN_OUTPUT_DIR", str(Path.cwd() / "results"))

CAPTURE24_DIRECTORY = os.environ.get("CAPTURE24_ROOT", str(Path.cwd() / "data" / "capture24"))

REQUIRE_REAL_MOTION = True

LOGGER = logging.getLogger("wban_final_experiment")

LOGGER.setLevel(logging.INFO)

LOGGER.propagate = False

if not LOGGER.handlers:
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
    LOGGER.addHandler(stream_handler)

os.makedirs(RESULTS_DIRECTORY, exist_ok=True)

OPERATION_MODES = ["PerDevice", "NaiveShared", "DeadlineShared"]

BOOTSTRAP_RESAMPLES = 10000

EXCEL_MAX_DATA_ROWS = 1000000

SIMULATION_DAYS = 60

SIMULATION_HOURS = SIMULATION_DAYS * 24

PEER_COLUMNS = ["vitals_peer", "accel_peer", "motion_peer", "peer_count"]

ORIGINAL_FEATURE_COLUMNS = list(FEATURE_COLUMNS)

FEATURE_COLUMNS = list(dict.fromkeys(FEATURE_COLUMNS + PEER_COLUMNS))

ABLATION_FEATURE_GROUPS["no_peer"] = PEER_COLUMNS

CAPTURE_MAXIMUM_FILES = None

REQUIRE_ALL_CAPTURE_PARTICIPANTS = True

EXPECTED_CAPTURE_PARTICIPANTS = 151

MOTION_CACHE_WORKBOOK = ""

FEATURE_COLUMNS = list(ORIGINAL_FEATURE_COLUMNS)

SIMULATION_DAYS = 30

SIMULATION_HOURS = 720

N_DEVICES = 1000

SERVICE_TARGET_H = 4.0

BOOTSTRAP_RESAMPLES = 20000

RESULTS_DIRECTORY = os.environ.get("WBAN_OUTPUT_DIR", str(Path.cwd() / "results"))

MAIN_REALIZATION_SEEDS = list(range(50000, 50020))

PRECISION_REALIZATION_SEEDS = list(range(50020, 50080))

ROBUSTNESS_REALIZATION_SEEDS = list(range(60000, 60020))

PAPER_TARGETS = [0.05, 0.1, 0.2]

RESULTS_INPUT_WORKBOOK = ""

OBSERVED_MISS_MATCH_TOLERANCE = 0.005

RUN_PUBLIC_LOG_ANCHOR = True

PUBLIC_LOG_CSV = ""

REQUIRE_REAL_MOTION = True


In [ ]:
def action_summary(actions, episodes, probabilities=None, target=EMERGENCY_MISS_TARGET):
    """Safety counts independent of synthetic counterfactual outcomes."""
    emergency = episodes.cause.to_numpy() == CAUSE_CLINICAL_EMERGENCY
    emergency_count = int(emergency.sum())
    detected = int(((actions == ACTION_NURSE) & emergency).sum())
    lower, upper = exact_binomial_interval(detected, emergency_count)
    missed_rate = 1 - detected / emergency_count if emergency_count else np.nan
    return dict(
        N_Episodes=len(episodes),
        N_Emergencies=emergency_count,
        N_FirstLineEmergencyEscalations=detected,
        N_MissedEmergencies=emergency_count - detected,
        Missed_Emerg=missed_rate,
        Recall=1 - missed_rate,
        Recall_CP_Lower=lower,
        Recall_CP_Upper=upper,
        RecallTargetViolated=float(missed_rate > target) if emergency_count else np.nan,
        N_NurseFirstNonEmergency=int(((actions == ACTION_NURSE) & ~emergency).sum()),
        N_NurseFirst=int((actions == ACTION_NURSE).sum()),
    )


def apply_common_gate(actions, probabilities, threshold):
    actions = np.asarray(actions, dtype=int).copy()
    if threshold is not None:
        actions[probabilities[:, CAUSE_CLINICAL_EMERGENCY] >= threshold] = ACTION_NURSE
    return actions


def bootstrap_interval(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return (np.nan, np.nan)
    generator = np.random.default_rng(20261101)
    samples = values[generator.integers(0, len(values), (BOOTSTRAP_RESAMPLES, len(values)))].mean(
        axis=1
    )
    return tuple((float(value) for value in np.quantile(samples, [0.025, 0.975])))


def chronological_boundaries(start_hour, end_hour):
    proportions = np.cumsum(
        [TRAIN_FRACTION, PROBABILITY_CALIBRATION_FRACTION, POLICY_TUNING_FRACTION]
    )
    if np.any(proportions <= 0) or proportions[-1] >= 1 or end_hour <= start_hour:
        raise ValueError("Splits must leave a nonempty test interval.")
    return start_hour + (end_hour - start_hour) * proportions


def closed_loop_dispatch(context, evaluated, score, capacity, discipline="risk_deadline"):
    """Closed-loop service/cancellation replay with exogenous fault arrivals.

    Readiness, service completion and observed recovery change queue state. The future
    failure-attempt stream remains exogenous; no claim of a fully regenerative fleet model.
    Labels are used only after service in outcome evaluation, never in priority ordering.
    """
    frame = evaluated["episodes"]
    selected = evaluated["actions"]
    indices = np.arange(len(frame))
    needed = evaluated["outcomes"]["N"][indices, selected]
    arrivals = frame.time.to_numpy() + OBSERVATION_DELAY_H + evaluated["deadline"]
    recovery = frame.time.to_numpy() + frame.D_nat.to_numpy()
    deadline = frame.time.to_numpy() + OBSERVATION_DELAY_H + SERVICE_TARGET_H
    requests = sorted(np.flatnonzero(needed), key=lambda index: (arrivals[index], int(index)))
    waiting = []
    completion = {}
    cancelled = []
    pending = []
    cursor = 0
    hour = int(np.floor(arrivals[requests[0]])) if requests else int(context["test_start"])
    max_slots = len(requests) if capacity is None else int(capacity)
    while cursor < len(requests) or pending:
        while cursor < len(requests) and arrivals[requests[cursor]] <= hour:
            pending.append(int(requests[cursor]))
            cursor += 1
        surviving = []
        for index in pending:
            if selected[index] != ACTION_NURSE and recovery[index] <= hour:
                cancelled.append(index)
            else:
                surviving.append(index)
        pending = surviving
        if not pending and cursor < len(requests):
            hour = max(hour, int(np.ceil(arrivals[requests[cursor]])))
            continue
        if discipline == "risk":
            pending.sort(key=lambda index: (-score[index], arrivals[index], index))
        elif discipline == "edf":
            pending.sort(key=lambda index: (deadline[index], -score[index], index))
        elif discipline == "risk_deadline":
            pending.sort(
                key=lambda index: (
                    -(score[index] * (1 + max(0.0, hour - deadline[index]))),
                    deadline[index],
                    index,
                )
            )
        else:
            raise ValueError("Unknown service discipline")
        for index in pending[:max_slots]:
            service_start = arrivals[index] if capacity is None else float(hour)
            completion[index] = service_start + DEFAULT_LOSS_PARAMETERS["lat_nurse"]
            waiting.append(service_start - arrivals[index])
        pending = pending[max_slots:]
        hour += 1
    emergency = frame.cause.eq(CAUSE_CLINICAL_EMERGENCY).to_numpy()
    response = np.full(len(frame), np.nan)
    for index, done in completion.items():
        response[index] = done - float(frame.time.iloc[index])
    emergency_response = response[emergency]
    finite = emergency_response[np.isfinite(emergency_response)]
    return dict(
        Capacity="Unlimited" if capacity is None else capacity,
        Discipline=discipline,
        Requests=len(requests),
        CancelledTechnicalRequests=len(cancelled),
        Served=len(completion),
        MeanQueueDelay_h=float(np.mean(waiting)) if waiting else np.nan,
        EmergencyMeanServiceCompletion_h=float(np.mean(finite)) if len(finite) else np.nan,
        EmergencyP95ServiceCompletion_h=float(np.quantile(finite, 0.95)) if len(finite) else np.nan,
        EmergencyServiceDeadlineMiss=(
            float(np.mean(finite > OBSERVATION_DELAY_H + SERVICE_TARGET_H))
            if len(finite)
            else np.nan
        ),
        UnservedEmergencies=int((emergency & ~np.isfinite(response)).sum()),
        BacklogAtHorizon=sum((done > context["test_end"] for done in completion.values())),
    )


def compute_action_outcomes(
    episodes, loss_parameters=DEFAULT_LOSS_PARAMETERS, wait_timeout_hours=None
):
    """Evaluate every action using episode-level shared random uniforms.

    Resolution time is measured from offline onset, including the common
    observation delay; natural recovery is evaluated after that delay. Costs
    use abstract operational units. This timing protocol differs from v3.
    Failed actions trigger nurse fallback after verification. Returns arrays
    for resolution time, action cost, nurse involvement and total loss.
    """
    wait_timeout_hours = (
        loss_parameters["wait_T"] if wait_timeout_hours is None else wait_timeout_hours
    )
    cause_labels, natural_recovery_hours = (
        episodes["cause"].values,
        np.maximum(0.0, episodes["D_nat"].values - OBSERVATION_DELAY_H),
    )
    action_uniforms = episodes[["U1", "U2", "U3"]].values
    emergency_mask = cause_labels == CAUSE_CLINICAL_EMERGENCY
    replacement_required = (cause_labels == CAUSE_HARDWARE_FAULT) & (
        episodes["U_hw"].values < loss_parameters["p_hw_needs_replace"]
    )
    fallback_resolution_hours = loss_parameters["lat_nurse"] + np.where(
        replacement_required, loss_parameters["lat_replace"], 0.0
    )
    fallback_costs = loss_parameters["cost"][ACTION_NURSE] + np.where(
        replacement_required, loss_parameters["cost"][ACTION_REPLACE], 0.0
    )
    episode_count = len(episodes)
    resolution_hours = np.zeros((episode_count, 5))
    action_costs = np.zeros((episode_count, 5))
    nurse_involvement = np.zeros((episode_count, 5), bool)
    timeout_mask = natural_recovery_hours > wait_timeout_hours
    resolution_hours[:, ACTION_WAIT] = np.where(
        timeout_mask, wait_timeout_hours + fallback_resolution_hours, natural_recovery_hours
    )
    action_costs[:, ACTION_WAIT] = np.where(timeout_mask, fallback_costs, 0.0)
    nurse_involvement[:, ACTION_WAIT] = timeout_mask
    resolution_hours[:, ACTION_NURSE] = np.minimum(
        natural_recovery_hours, fallback_resolution_hours
    )
    action_costs[:, ACTION_NURSE] = fallback_costs
    nurse_involvement[:, ACTION_NURSE] = True
    for (
        action_index,
        success_probability_key,
        action_latency,
        verification_hours,
        uniform_index,
    ) in (
        (ACTION_NUDGE, "p_nudge", loss_parameters["lat_nudge"], loss_parameters["ver_nudge"], 0),
        (
            ACTION_RECONNECT,
            "p_recon",
            loss_parameters["lat_recon"],
            loss_parameters["ver_recon"],
            1,
        ),
        (
            ACTION_REPLACE,
            "p_repl",
            loss_parameters["lat_replace"],
            loss_parameters["ver_replace"],
            2,
        ),
    ):
        action_success = (
            action_uniforms[:, uniform_index]
            < loss_parameters[success_probability_key][cause_labels]
        )
        fallback_mask = ~action_success & (natural_recovery_hours > verification_hours)
        resolution_hours[:, action_index] = np.where(
            action_success,
            np.minimum(natural_recovery_hours, action_latency),
            np.where(
                natural_recovery_hours > verification_hours,
                np.minimum(natural_recovery_hours, verification_hours + fallback_resolution_hours),
                natural_recovery_hours,
            ),
        )
        action_costs[:, action_index] = loss_parameters["cost"][action_index] + np.where(
            fallback_mask, fallback_costs, 0.0
        )
        nurse_involvement[:, action_index] = fallback_mask
    resolution_hours += OBSERVATION_DELAY_H
    hourly_loss_weights = (
        loss_parameters["lam_data"] + loss_parameters["lam_emerg"] * emergency_mask
    )
    return dict(
        T=resolution_hours,
        C=action_costs,
        N=nurse_involvement,
        loss=action_costs + hourly_loss_weights[:, None] * resolution_hours,
    )


In [ ]:
def compute_policy_metrics(actions, outcomes, episodes, device_days, loss_parameters):
    indices = np.arange(len(episodes))
    resolution = outcomes["T"][indices, actions]
    costs = outcomes["C"][indices, actions]
    nurses = outcomes["N"][indices, actions]
    emergency = episodes.cause.to_numpy() == CAUSE_CLINICAL_EMERGENCY
    summary = action_summary(actions, episodes)
    summary.update(
        Mean_Loss=outcomes["loss"][indices, actions].mean(),
        Cost_Action=costs.mean(),
        Cost_Data=loss_parameters["lam_data"] * resolution.mean(),
        Cost_Clinical=loss_parameters["lam_emerg"] * (resolution * emergency).mean(),
        Cost_Infrastructure=0.0,
        Mean_TTR=resolution.mean(),
        P95_TTR=np.quantile(resolution, 0.95),
        Emerg_TTR=resolution[emergency].mean() if emergency.any() else np.nan,
        Offline_h_per_device_day=resolution.sum() / device_days,
        Nurse_per_1000dd=nurses.sum() / device_days * 1000,
        NurseFirst_nonEm_per_1000dd=summary["N_NurseFirstNonEmergency"] / device_days * 1000,
        Immediate_Device_Actions=int((actions != ACTION_WAIT).sum()),
    )
    return summary


def detect_gateway_incidents(
    observations, opening_threshold=0.5, closing_threshold=0.2, clear_windows=2
):
    """Open/update/close persistent tickets using telemetry aggregates only.

    Observations describe a completed hour; detection time is hour+1. A ticket
    closes after consecutive low-offline windows. Open tickets are right
    censored at the observation horizon. No simulator cause/incident IDs enter
    this detector. All policies share the same detector and repair settings.
    """
    columns = ["ticket_id", "gateway_id", "detected_hour", "closed_hour", "right_censored"]
    records = []
    for gateway, group in observations.groupby("gateway_id", sort=True):
        opened = None
        consecutive_clear = 0
        for row in group.sort_values("hour").itertuples():
            fraction = row.offline_devices / max(row.total_devices, 1)
            observation_time = float(row.hour) + 1.0
            if opened is None and fraction > opening_threshold:
                opened = dict(
                    ticket_id=len(records), gateway_id=int(gateway), detected_hour=observation_time
                )
                records.append(opened)
                consecutive_clear = 0
            elif opened is not None:
                consecutive_clear = consecutive_clear + 1 if fraction <= closing_threshold else 0
                if consecutive_clear >= clear_windows:
                    opened.update(closed_hour=observation_time, right_censored=False)
                    opened = None
                    consecutive_clear = 0
        if opened is not None:
            opened.update(closed_hour=float(group.hour.max()) + 1.0, right_censored=True)
    return pd.DataFrame(records, columns=columns)


def estimate_loss_matrix(training_episodes, training_losses):
    """Estimate cause-by-action mean loss from training episodes only.

    This estimator uses full simulated counterfactual outcomes. It does not
    learn from chosen-action-only operational logs. Absent causes use the
    pooled training mean; no test outcomes enter the estimated matrix.
    """
    cause_labels = training_episodes["cause"].values
    pooled_mean_losses = training_losses.mean(0)
    return np.vstack(
        [
            (
                training_losses[cause_labels == cause_index].mean(0)
                if (cause_labels == cause_index).any()
                else pooled_mean_losses
            )
            for cause_index in range(N_CAUSES)
        ]
    )


def evaluate_incident_detection(truth, tickets, test_start, test_end):
    """Match openings to known injected faults; report duplicates, merges and censoring."""
    visible_truth = truth[(truth.recovery_hour > test_start) & (truth.onset_hour < test_end)].copy()
    eligible = visible_truth[visible_truth.onset_hour >= test_start]
    matches = []
    false_tickets = 0
    merged_tickets = 0
    for ticket in tickets.itertuples():
        opening_matches = visible_truth[
            (visible_truth.gateway_id == ticket.gateway_id)
            & (visible_truth.onset_hour < ticket.detected_hour)
            & (visible_truth.recovery_hour > ticket.detected_hour - 1.0)
        ]
        overlapping = visible_truth[
            (visible_truth.gateway_id == ticket.gateway_id)
            & (visible_truth.onset_hour < ticket.closed_hour)
            & (visible_truth.recovery_hour > ticket.detected_hour - 1.0)
        ]
        false_tickets += int(opening_matches.empty)
        merged_tickets += int(len(overlapping) > 1)
        for fault in opening_matches.itertuples():
            matches.append(
                dict(
                    incident_id=int(fault.incident_id),
                    ticket_id=int(ticket.ticket_id),
                    detected_hour=float(ticket.detected_hour),
                    detection_delay_h=float(ticket.detected_hour - fault.onset_hour),
                )
            )
    matched = pd.DataFrame(
        matches, columns=["incident_id", "ticket_id", "detected_hour", "detection_delay_h"]
    )
    eligible_ids = set(eligible.incident_id)
    detected_ids = set(matched.incident_id) & eligible_ids
    eligible_matches = matched[matched.incident_id.isin(eligible_ids)]
    first = eligible_matches.sort_values("detected_hour").drop_duplicates("incident_id")
    summary = dict(
        Faults=len(eligible),
        Detected_Faults=len(detected_ids),
        Missed_Faults=len(eligible) - len(detected_ids),
        LeftCensored_Faults=len(visible_truth) - len(eligible),
        Tickets=len(tickets),
        False_Tickets=false_tickets,
        Duplicate_Tickets=len(eligible_matches) - len(first),
        Merged_Tickets=merged_tickets,
        Detection_Rate=len(detected_ids) / len(eligible) if len(eligible) else np.nan,
        Mean_TTD_h_detected=first.detection_delay_h.mean() if len(first) else np.nan,
        P95_TTD_h_detected=first.detection_delay_h.quantile(0.95) if len(first) else np.nan,
    )
    return (summary, matched)


def evaluate_incident_policy(
    context,
    actions,
    mode,
    loss_parameters=DEFAULT_LOSS_PARAMETERS,
    repair_cost=INFRASTRUCTURE_REPAIR_COST,
    repair_delay=INFRASTRUCTURE_REPAIR_DELAY_H,
    wait_timeout=None,
):
    """Deadline-preserving aggregation candidate. No true cause is used to select actions.

    The guard preserves a scheduled fallback deadline, not clinical safety under arbitrary dynamics.
    For true emergencies the simulator assumes non-nurse interventions cannot cure the emergency.
    The same guard and shared repair are offered to every operational baseline.
    """
    episodes = context["test"].copy()
    selected = np.asarray(actions, dtype=int).copy()
    before_deadline = initial_fallback_deadlines(actions, loss_parameters, wait_timeout)
    wait_deadline = np.full(
        len(episodes),
        loss_parameters["wait_T"] if wait_timeout is None else wait_timeout,
        dtype=float,
    )
    eligible = ticket_membership(context) & (selected != ACTION_NURSE) & (selected != ACTION_WAIT)
    eligible = eligible & ~np.asarray(
        context.get("suppression_protected", np.zeros(len(episodes), dtype=bool))
    )
    suppressed = eligible if mode != "PerDevice" else np.zeros(len(episodes), dtype=bool)
    if mode not in OPERATION_MODES:
        raise ValueError("Unknown infrastructure mode.")
    selected[suppressed] = ACTION_WAIT
    if mode == "DeadlineShared":
        wait_deadline[suppressed] = np.minimum(
            wait_deadline[suppressed], before_deadline[suppressed]
        )
    if mode != "PerDevice":
        first = context["matches"].sort_values("detected_hour").drop_duplicates("incident_id")
        for match in first.itertuples():
            affected = (
                episodes.infrastructure_incident_id.to_numpy() == match.incident_id
            ) & episodes.cause.eq(CAUSE_GATEWAY_OUTAGE).to_numpy()
            remaining = np.maximum(
                0.0, match.detected_hour + repair_delay - episodes.time.to_numpy()[affected]
            )
            episodes.loc[affected, "D_nat"] = np.minimum(
                episodes.loc[affected, "D_nat"].to_numpy(), remaining
            )
    outcomes = compute_action_outcomes(episodes, loss_parameters, wait_timeout_hours=wait_deadline)
    metrics = compute_policy_metrics(
        selected, outcomes, episodes, context["test_device_days"], loss_parameters
    )
    infra_cost = (
        len(context["tickets"]) * (INFRASTRUCTURE_TICKET_COST + repair_cost)
        if mode != "PerDevice"
        else 0.0
    )
    metrics["Mean_Loss"] += infra_cost / len(episodes)
    metrics["Cost_Infrastructure"] = infra_cost / len(episodes)
    emergency = episodes.cause.eq(CAUSE_CLINICAL_EMERGENCY).to_numpy()
    after_deadline = initial_fallback_deadlines(selected, loss_parameters, wait_timeout)
    after_deadline[selected == ACTION_WAIT] = wait_deadline[selected == ACTION_WAIT]
    metrics.update(
        ActionsSuppressed=int(suppressed.sum()),
        EmergencyActionsSuppressed=int((suppressed & emergency).sum()),
        UnrelatedActionsSuppressed=int(
            (suppressed & ~episodes.cause.eq(CAUSE_GATEWAY_OUTAGE).to_numpy()).sum()
        ),
        DeadlineExtensions=int((after_deadline > before_deadline + 1e-09).sum()),
        EmergencyDeadlineExtensions=int(
            ((after_deadline > before_deadline + 1e-09) & emergency).sum()
        ),
        TicketCostTotal=float(infra_cost),
    )
    if mode == "DeadlineShared" and metrics["DeadlineExtensions"]:
        raise AssertionError("Deadline guard extended a scheduled fallback deadline.")
    return dict(
        metrics=metrics,
        actions=selected,
        outcomes=outcomes,
        episodes=episodes,
        before_deadline=before_deadline,
        deadline=after_deadline,
        suppressed=suppressed,
    )


def exact_binomial_interval(successes, trials, confidence=0.95):
    """Clopper--Pearson interval, descriptive when episodes are correlated."""
    if trials == 0:
        return (np.nan, np.nan)
    tail = (1 - confidence) / 2
    lower = 0.0 if successes == 0 else stats.beta.ppf(tail, successes, trials - successes + 1)
    upper = (
        1.0 if successes == trials else stats.beta.ppf(1 - tail, successes + 1, trials - successes)
    )
    return (float(lower), float(upper))


In [ ]:
def expert_rule_actions(
    episodes, vitals_threshold=2.0, acceleration_threshold=0.45, missing_aware=False
):
    """Original rule structure with explicit, tunable clinical trigger thresholds."""
    actions = np.full(len(episodes), ACTION_RECONNECT)
    infrastructure = (episodes.cohort_gw > 0.5).to_numpy()
    low_battery = (episodes.batt_level < 8).to_numpy()
    actions[infrastructure] = ACTION_WAIT
    actions[(episodes.contact_q < 0.4).to_numpy() | low_battery] = ACTION_NUDGE
    clinical = (
        (episodes.vitals_anom > vitals_threshold) | (episodes.accel_peak > acceleration_threshold)
    ).to_numpy()
    actions[clinical] = ACTION_NURSE
    if missing_aware:
        missing = episodes[["vitals_anom", "accel_peak"]].isna().any(axis=1).to_numpy()
        actions[missing & ~(infrastructure | low_battery)] = ACTION_NURSE
    return actions


def export_results_workbook(tables, protocol, destination):
    import tempfile
    import importlib.util

    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    use_xlsxwriter = importlib.util.find_spec("xlsxwriter") is not None
    if not use_xlsxwriter and importlib.util.find_spec("openpyxl") is None:
        raise ImportError("Install xlsxwriter or openpyxl before executing the study.")
    with tempfile.TemporaryDirectory(prefix=".wban_export_", dir=destination.parent) as temporary:
        temporary_file = Path(temporary) / "results.xlsx"
        if use_xlsxwriter:
            import xlsxwriter

            book = xlsxwriter.Workbook(str(temporary_file), {"constant_memory": True})
            header = book.add_format(
                {"bold": True, "font_color": "white", "bg_color": "#17365D", "text_wrap": True}
            )
            number = book.add_format({"num_format": "0.0000"})
            integer = book.add_format({"num_format": "0"})
            wrapped = book.add_format({"text_wrap": True, "valign": "top"})
        else:
            from openpyxl import Workbook

            book = Workbook(write_only=True)
        used_names = set()
        index = []

        def write_sheet(name, rows, columns):
            base = name[:27]
            name = base
            suffix = 1
            while name in used_names:
                name = base[:26] + "_" + str(suffix)
                suffix += 1
            used_names.add(name)
            if use_xlsxwriter:
                sheet = book.add_worksheet(name)
                sheet.freeze_panes(1, 0)
                sheet.set_row(0, 32)
                sheet.set_column(0, len(columns) - 1, 19)
                if name in ["Overview", "Protocol"]:
                    sheet.set_column(0, 0, 34)
                    sheet.set_column(1, 1, 100, wrapped)
                sheet.write_row(0, 0, columns, header)
                for row_number, row in enumerate(rows, 1):
                    for column_number, key in enumerate(columns):
                        value = workbook_value(row.get(key))
                        if isinstance(value, (int, float)) and (not isinstance(value, bool)):
                            sheet.write_number(
                                row_number,
                                column_number,
                                value,
                                integer if isinstance(value, int) else number,
                            )
                        else:
                            sheet.write(row_number, column_number, value)
                if rows:
                    sheet.autofilter(0, 0, len(rows), len(columns) - 1)
            else:
                sheet = book.create_sheet(name)
                sheet.freeze_panes = "A2"
                sheet.append(columns)
                for row in rows:
                    sheet.append([workbook_value(row.get(key)) for key in columns])
            index.append(dict(Table=name, Rows=len(rows), Columns=len(columns)))

        overview = [
            dict(Item="Study", Value="Cohort-aware fault management in WBAN fleets"),
            dict(Item="Status", Value=protocol.get("status", "running")),
            dict(
                Item="Primary setting",
                Value="Reconstructed PDF protocol; static emergency gate; cohort ablation",
            ),
            dict(
                Item="Output contract",
                Value="One XLSX workbook plus vector PDF figures; no CSV files",
            ),
            dict(
                Item="Safety interpretation",
                Value="Scheduled fallback preservation is not a clinical guarantee",
            ),
            dict(
                Item="Calibration",
                Value="Conditional on declared environment, fixed score, iid positives; not arbitrary shift",
            ),
            dict(Item="Empty cells", Value="Unavailable/not applicable; never interpret as zero"),
            dict(
                Item="Statistics",
                Value="Paired seed bootstrap and Wilcoxon; correction families stated in tables",
            ),
            dict(
                Item="Pre-registration",
                Value="Fixed before execution by code; no public preregistration is claimed",
            ),
        ]
        write_sheet("Overview", overview, ["Item", "Value"])
        write_sheet(
            "Protocol",
            [dict(Parameter=k, Value=v) for k, v in protocol.items()],
            ["Parameter", "Value"],
        )
        for name, records in tables.items():
            records = records.to_dict("records") if isinstance(records, pd.DataFrame) else records
            if not records:
                continue
            columns = list(dict.fromkeys((key for row in records for key in row)))
            for part, start in enumerate(range(0, len(records), EXCEL_MAX_DATA_ROWS), 1):
                write_sheet(
                    name if len(records) <= EXCEL_MAX_DATA_ROWS else name + "_" + str(part),
                    records[start : start + EXCEL_MAX_DATA_ROWS],
                    columns,
                )
        write_sheet("TableIndex", list(index), ["Table", "Rows", "Columns"])
        if use_xlsxwriter:
            book.close()
        else:
            book.save(temporary_file)
        os.replace(temporary_file, destination)


def fit_emergency_model(training, features, seed):
    labels = training.cause.eq(CAUSE_CLINICAL_EMERGENCY).to_numpy(dtype=int)
    counts = np.bincount(labels, minlength=2)
    if counts.min() == 0:
        raise ValueError("Both emergency and non-emergency training examples are required.")
    model = xgb.XGBClassifier(
        **CLASSIFIER_PARAMETERS,
        objective="binary:logistic",
        tree_method="hist",
        n_jobs=MODEL_THREADS,
        random_state=seed
    )
    model.fit(
        training[features],
        labels,
        sample_weight=np.where(labels == 1, np.sqrt(counts[0] / counts[1]), 1.0),
    )
    return dict(model=model, features=features)


def fit_posteriors(
    training, probability_calibration, policy_tuning, test, excluded_features=(), seed=0
):
    """Fit only on training; isotonic fitting and gate tuning use separate windows."""
    feature_columns = [column for column in FEATURE_COLUMNS if column not in excluded_features]
    observed_causes = np.unique(training.cause.to_numpy(dtype=int))
    if len(observed_causes) < 2:
        raise ValueError("At least two causes are required to fit the classifier.")
    encoded_labels = np.searchsorted(observed_causes, training.cause.to_numpy(dtype=int))
    class_counts = np.bincount(encoded_labels)
    weights = (len(training) / (len(observed_causes) * class_counts)) ** CLASS_WEIGHT_EXPONENT
    objective = "binary:logistic" if len(observed_causes) == 2 else "multi:softprob"
    classifier = xgb.XGBClassifier(
        **CLASSIFIER_PARAMETERS,
        objective=objective,
        tree_method="hist",
        n_jobs=MODEL_THREADS,
        random_state=seed
    )
    classifier.fit(training[feature_columns], encoded_labels, sample_weight=weights[encoded_labels])
    fitted = dict(
        classifier=classifier,
        observed_causes=observed_causes,
        feature_columns=feature_columns,
        calibrators={},
        seed=seed,
    )
    calibration_raw = predict_posteriors(fitted, probability_calibration, calibrated=False)
    for cause_index in observed_causes:
        binary_labels = (probability_calibration.cause.to_numpy() == cause_index).astype(int)
        if binary_labels.sum() >= 5 and (1 - binary_labels).sum() >= 5:
            fitted["calibrators"][int(cause_index)] = IsotonicRegression(
                y_min=0, y_max=1, out_of_bounds="clip"
            ).fit(calibration_raw[:, cause_index], binary_labels)
    tuning_raw = predict_posteriors(fitted, policy_tuning, calibrated=False)
    tuning_calibrated = predict_posteriors(fitted, policy_tuning)
    emergency_mask = policy_tuning.cause.to_numpy() == CAUSE_CLINICAL_EMERGENCY
    threshold = heuristic_emergency_threshold(
        tuning_calibrated[emergency_mask, CAUSE_CLINICAL_EMERGENCY], EMERGENCY_MISS_TARGET
    )
    tolerance_threshold, tolerance_status = tolerance_emergency_threshold(
        tuning_calibrated[emergency_mask, CAUSE_CLINICAL_EMERGENCY], EMERGENCY_MISS_TARGET
    )
    fitted.update(
        tuning_raw=tuning_raw,
        tuning=tuning_calibrated,
        raw=predict_posteriors(fitted, test, calibrated=False),
        cal=predict_posteriors(fitted, test),
        tau_cal=threshold,
        tau_raw=heuristic_emergency_threshold(
            tuning_raw[emergency_mask, CAUSE_CLINICAL_EMERGENCY], EMERGENCY_MISS_TARGET
        ),
        tolerance_threshold=tolerance_threshold,
        tolerance_status=tolerance_status,
        positive_scores=tuning_calibrated[emergency_mask, CAUSE_CLINICAL_EMERGENCY],
        tuning_positive_count=int(emergency_mask.sum()),
        probability_calibration_positive_count=int(
            (probability_calibration.cause == CAUSE_CLINICAL_EMERGENCY).sum()
        ),
    )
    return fitted


def heuristic_emergency_threshold(positive_scores, miss_target):
    """Retain the sparse-positive fallback as an explicitly heuristic comparator."""
    scores = np.sort(np.asarray(positive_scores, dtype=float))
    if len(scores) < 15:
        return 0.05
    rank = int(np.floor(miss_target * (len(scores) + 1)))
    threshold = scores[rank - 1] if rank >= 1 else scores[0] * 0.5
    return float(np.clip(threshold, 0.01, 0.3))


def incident_context(fleet, test, seed):
    boundaries = chronological_boundaries(0.0, SIMULATION_HOURS)
    context = dict(
        test=test,
        seed=seed,
        test_start=float(boundaries[-1]),
        test_end=SIMULATION_HOURS,
        test_device_days=N_DEVICES * (SIMULATION_HOURS - boundaries[-1]) / 24,
        truth=fleet.attrs["gateway_incidents"],
    )
    observations = fleet.attrs["gateway_observations"]
    observations = observations[
        (observations.hour >= context["test_start"]) & (observations.hour < context["test_end"])
    ]
    context["observations"] = observations
    tickets = detect_gateway_incidents(observations)
    detection, matches = evaluate_incident_detection(
        context["truth"], tickets, context["test_start"], context["test_end"]
    )
    context.update(tickets=tickets, matches=matches, detection=detection)
    return context


In [ ]:
def initial_fallback_deadlines(actions, loss_parameters, wait_timeout=None):
    default_wait = loss_parameters["wait_T"] if wait_timeout is None else wait_timeout
    lookup = np.array(
        [
            default_wait,
            loss_parameters["ver_nudge"],
            loss_parameters["ver_recon"],
            0.0,
            loss_parameters["ver_replace"],
        ],
        dtype=float,
    )
    return lookup[np.asarray(actions, dtype=int)]


def initial_operational_actions(episodes, probabilities, loss_matrix, rule_parameters):
    gateway = episodes.cohort_gw.to_numpy() > 0.5
    simple = np.full(len(episodes), ACTION_RECONNECT, dtype=int)
    simple[(episodes.batt_level.to_numpy() < 8) | (episodes.contact_q.to_numpy() < 0.4)] = (
        ACTION_NUDGE
    )
    simple[gateway] = ACTION_WAIT
    individual_view = episodes.copy()
    individual_view["cohort_gw"] = 0.0
    per_device = expert_rule_actions(individual_view, missing_aware=True)
    return dict(
        CausePolicy=select_actions(probabilities, loss_matrix),
        TunedRule=expert_rule_actions(episodes, **rule_parameters),
        GatewayRule=simple,
        PerDeviceRule=per_device,
        AlwaysNurse=np.full(len(episodes), ACTION_NURSE),
        Timeout1h=np.full(len(episodes), ACTION_WAIT),
        Timeout4h=np.full(len(episodes), ACTION_WAIT),
        Timeout12h=np.full(len(episodes), ACTION_WAIT),
    )


def normalize_probabilities(probabilities, probability_floor=1e-06):
    """Floor and normalize each multiclass posterior row."""
    probabilities = np.clip(probabilities, probability_floor, None)
    return probabilities / probabilities.sum(axis=1, keepdims=True)


def predict_posteriors(fitted, episodes, calibrated=True):
    """Predict using frozen artifacts; this function never fits on test labels."""
    probabilities = np.full((len(episodes), N_CAUSES), 1e-09)
    probabilities[:, fitted["observed_causes"]] = fitted["classifier"].predict_proba(
        episodes[fitted["feature_columns"]]
    )
    probabilities = normalize_probabilities(probabilities)
    if calibrated:
        for cause_index, calibrator in fitted["calibrators"].items():
            probabilities[:, cause_index] = calibrator.predict(probabilities[:, cause_index])
        probabilities = normalize_probabilities(probabilities)
    return probabilities


def rule_candidates(episodes, missing_aware=False):
    """Finite, prespecified clinical-threshold search; feature structure fixed."""
    for vitals in RULE_VITALS_THRESHOLDS:
        for acceleration in RULE_ACCELERATION_THRESHOLDS:
            parameters = dict(
                vitals_threshold=float(vitals),
                acceleration_threshold=float(acceleration),
                missing_aware=missing_aware,
            )
            yield (parameters, expert_rule_actions(episodes, **parameters))


def select_actions(probabilities, loss_matrix, threshold=None, mode="expected"):
    """Select an action, then apply the same escalation override to any policy."""
    actions = (
        (probabilities @ loss_matrix).argmin(axis=1)
        if mode == "expected"
        else CAUSE_TO_GREEDY_ACTION[probabilities.argmax(axis=1)]
    )
    return apply_common_gate(actions, probabilities, threshold)


In [ ]:
def serializable_metadata(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Unsupported metadata type: {type(value).__name__}")


def simulate_fleet_reference(
    seed,
    device_count=N_DEVICES,
    gateway_outage_probability=0.002,
    noise_multiplier=1.0,
    missing_sensor_probability=0.1,
    fatigue_multiplier=1.0,
    emergency_probability=0.0002,
    test_emergency_multiplier=1.0,
    burst_multiplier=1.0,
    motion_reference=None,
    emergency_fall_probability=0.6,
    emergency_stillness_probability=0.45,
    emergency_vitals_shift=2.0,
    outage_emergency_probability=0.0,
    gateway_count=5,
    gateway_failure_participation=1.0,
):
    """Simulate policy-independent offline episodes at hourly resolution.

    Each episode includes telemetry, cause, natural recovery and shared random
    uniforms for paired counterfactual comparisons. Cohort features contain
    only other devices' concurrent or preceding-three-hour offline counts.
    Emergency fall, stillness and vital signatures are synthetic assumptions.
    Non-wear motion centered at -2.7 log10(g) is assumed, not measured.
    Persistent gateway outages have truth IDs used only for evaluation.
    Gateway observation aggregates are sampled at the end of each hour.
    Short episodes resolved before observation do not trigger actions.
    Future episode arrivals remain policy-independent (open loop).
    """
    random_generator = np.random.default_rng(seed)
    fleet_size = device_count
    gateway_assignments, os_assignments, phone_model_assignments = (
        random_generator.integers(0, gateway_count, fleet_size),
        random_generator.integers(0, 3, fleet_size),
        random_generator.integers(0, 4, fleet_size),
    )
    charging_hours = random_generator.choice([20, 22, 23], fleet_size)
    battery_drain = random_generator.uniform(1.5, 3.0, fleet_size)
    charging_probabilities = random_generator.beta(17, 3, fleet_size)
    battery = random_generator.uniform(40, 100, fleet_size)
    hours_since_charge = random_generator.uniform(0, 24, fleet_size)
    nonwear_base_hazard = random_generator.gamma(4.0, 0.003 / 4.0, fleet_size)
    bluetooth_multipliers = np.array([0.5, 1.0, 1.5, 3.0])[phone_model_assignments]
    os_multipliers = np.array([0.5, 1.0, 2.0])[os_assignments]
    next_online_hours = np.zeros(fleet_size)
    previous_episode_counts = np.zeros(fleet_size)
    gateway_recovery_hours = np.zeros(gateway_count)
    gateway_incident_ids = np.full(gateway_count, -1, dtype=int)
    incident_records = []
    gateway_observation_records = []
    next_incident_id = 0
    test_start = SIMULATION_HOURS * (TRAIN_FRACTION + CALIBRATION_FRACTION)
    cohort_sizes = dict(
        gw=np.bincount(gateway_assignments, minlength=gateway_count),
        os=np.bincount(os_assignments, minlength=3),
        model=np.bincount(phone_model_assignments, minlength=4),
    )
    episode_columns = {
        column_name: []
        for column_name in [
            "time",
            "device",
            "cause",
            "D_nat",
            "infrastructure_incident_id",
            "batt_level",
            "batt_slope",
            "hrs_since_charge",
            "motion_E",
            "accel_peak",
            "contact_q",
            "rssi_trend",
            "vitals_anom",
            "prior_eps",
            "time_of_day",
            "gateway_id",
            "os_version",
            "phone_model",
            "U1",
            "U2",
            "U3",
            "U_hw",
        ]
    }
    cause_precedence = [
        CAUSE_CLINICAL_EMERGENCY,
        CAUSE_DEAD_BATTERY,
        CAUSE_GATEWAY_OUTAGE,
        CAUSE_HARDWARE_FAULT,
        CAUSE_APP_TERMINATION,
        CAUSE_BLUETOOTH_DROP,
        CAUSE_NONWEAR,
    ]

    def sample_worn_motion(sample_count):
        return (
            random_generator.choice(motion_reference, sample_count)
            if motion_reference is not None
            else random_generator.normal(-1.6, 0.5, sample_count)
        )

    def append_gateway_observations(hour):
        silent = (next_online_hours > hour + OBSERVATION_DELAY_H) | (
            gateway_recovery_hours[gateway_assignments] > hour + OBSERVATION_DELAY_H
        )
        offline_counts = np.bincount(
            gateway_assignments, weights=silent.astype(int), minlength=gateway_count
        )
        for gateway in range(gateway_count):
            gateway_observation_records.append(
                dict(
                    hour=hour,
                    gateway_id=gateway,
                    offline_devices=int(offline_counts[gateway]),
                    total_devices=int(cohort_sizes["gw"][gateway]),
                )
            )

    for hour in range(SIMULATION_HOURS):
        hour_of_day = hour % 24
        online = next_online_hours <= hour
        battery = np.where(online, battery - battery_drain, battery)
        hours_since_charge += 1
        charging_mask = (charging_hours == hour_of_day) & (
            random_generator.random(fleet_size) < charging_probabilities
        )
        battery[charging_mask] = 100.0
        hours_since_charge[charging_mask] = 0.0
        gateway_outage_mask = (
            random_generator.random(gateway_count) < gateway_outage_probability * burst_multiplier
        )
        os_incident_mask = random_generator.random(3) < 0.0015 * burst_multiplier
        model_incident_mask = random_generator.random(4) < 0.001 * burst_multiplier
        gateway_outage_durations = random_generator.lognormal(np.log(2.5), 0.7, gateway_count)
        newly_failed = gateway_outage_mask & (gateway_recovery_hours <= hour)
        for gateway in np.flatnonzero(newly_failed):
            gateway_recovery_hours[gateway] = hour + gateway_outage_durations[gateway]
            gateway_incident_ids[gateway] = next_incident_id
            incident_records.append(
                dict(
                    incident_id=next_incident_id,
                    gateway_id=int(gateway),
                    onset_hour=float(hour),
                    recovery_hour=float(gateway_recovery_hours[gateway]),
                    registered_devices=int(cohort_sizes["gw"][gateway]),
                )
            )
            next_incident_id += 1
        gateway_outage_mask = gateway_recovery_hours > hour
        gateway_outage_durations = np.maximum(0.0, gateway_recovery_hours - hour)
        event_uniforms = random_generator.random((fleet_size, 6))
        current_emergency_probability = emergency_probability * (
            test_emergency_multiplier if hour >= test_start else 1.0
        )
        nonwear_hazard = (
            nonwear_base_hazard
            * (1 + 1.5 * fatigue_multiplier * hour / SIMULATION_HOURS)
            * (1.3 if 18 <= hour_of_day <= 23 else 1.0)
        )
        cause_triggers = {
            CAUSE_CLINICAL_EMERGENCY: event_uniforms[:, 0]
            < np.where(
                newly_failed[gateway_assignments],
                np.maximum(current_emergency_probability, outage_emergency_probability),
                current_emergency_probability,
            ),
            CAUSE_DEAD_BATTERY: battery <= 0,
            CAUSE_GATEWAY_OUTAGE: gateway_outage_mask[gateway_assignments]
            & (random_generator.random(fleet_size) < gateway_failure_participation),
            CAUSE_HARDWARE_FAULT: event_uniforms[:, 1] < 5e-05,
            CAUSE_APP_TERMINATION: os_incident_mask[os_assignments] & (event_uniforms[:, 2] < 0.4)
            | (event_uniforms[:, 3] < 0.0002 * os_multipliers),
            CAUSE_BLUETOOTH_DROP: model_incident_mask[phone_model_assignments]
            & (event_uniforms[:, 4] < 0.25)
            | (random_generator.random(fleet_size) < 0.0004 * bluetooth_multipliers),
            CAUSE_NONWEAR: event_uniforms[:, 5] < nonwear_hazard,
        }
        cause_assignments = np.full(fleet_size, -1)
        for cause_index in reversed(cause_precedence):
            cause_assignments = np.where(
                cause_triggers[cause_index], cause_index, cause_assignments
            )
        cause_assignments = np.where(online, cause_assignments, -1)
        episode_device_indices = np.flatnonzero(cause_assignments >= 0)
        if len(episode_device_indices) == 0:
            append_gateway_observations(hour)
            continue
        episode_count = len(episode_device_indices)
        episode_causes = cause_assignments[episode_device_indices]
        natural_recovery_hours = np.full(episode_count, np.inf)
        natural_recovery_hours[episode_causes == CAUSE_DEAD_BATTERY] = (
            2 + random_generator.exponential(10, (episode_causes == CAUSE_DEAD_BATTERY).sum())
        )
        natural_recovery_hours[episode_causes == CAUSE_GATEWAY_OUTAGE] = gateway_outage_durations[
            gateway_assignments[episode_device_indices][episode_causes == CAUSE_GATEWAY_OUTAGE]
        ]
        natural_recovery_hours[episode_causes == CAUSE_APP_TERMINATION] = (
            random_generator.lognormal(
                np.log(18), 0.6, (episode_causes == CAUSE_APP_TERMINATION).sum()
            )
        )
        natural_recovery_hours[episode_causes == CAUSE_BLUETOOTH_DROP] = (
            0.25 + random_generator.exponential(2.5, (episode_causes == CAUSE_BLUETOOTH_DROP).sum())
        )
        natural_recovery_hours[episode_causes == CAUSE_NONWEAR] = random_generator.lognormal(
            np.log(8), 0.8, (episode_causes == CAUSE_NONWEAR).sum()
        )
        next_online_hours[episode_device_indices] = hour + np.where(
            np.isfinite(natural_recovery_hours), np.minimum(natural_recovery_hours, 48), 24.0
        )
        append_gateway_observations(hour)
        emergency_mask, nonwear_mask, bluetooth_mask, hardware_mask = (
            episode_causes == CAUSE_CLINICAL_EMERGENCY,
            episode_causes == CAUSE_NONWEAR,
            episode_causes == CAUSE_BLUETOOTH_DROP,
            episode_causes == CAUSE_HARDWARE_FAULT,
        )
        motion_energy = sample_worn_motion(episode_count) + random_generator.normal(
            0, 0.1 * noise_multiplier, episode_count
        )
        acceleration_peak = random_generator.normal(
            0.0, 0.25, episode_count
        ) + random_generator.normal(0, 0.2 * (noise_multiplier - 1), episode_count)
        contact_quality = random_generator.normal(
            0.85, 0.08, episode_count
        ) + random_generator.normal(0, 0.03 * noise_multiplier, episode_count)
        vitals_anomaly = random_generator.normal(0.0, 1.0, episode_count) + random_generator.normal(
            0, 0.5 * (noise_multiplier - 1), episode_count
        )
        rssi = random_generator.normal(0, 2 * noise_multiplier, episode_count)
        battery_slope = -battery_drain[episode_device_indices] + random_generator.normal(
            0, 0.3 * noise_multiplier, episode_count
        )
        battery_level = np.clip(
            battery[episode_device_indices] + battery_drain[episode_device_indices], 0, 100
        ) + random_generator.normal(0, 2 * noise_multiplier, episode_count)
        signature_mask = emergency_mask & (
            random_generator.random(episode_count) < emergency_stillness_probability
        )
        motion_energy[signature_mask] = random_generator.normal(-2.2, 0.3, signature_mask.sum())
        signature_mask = emergency_mask & (
            random_generator.random(episode_count) < emergency_fall_probability
        )
        acceleration_peak[signature_mask] = random_generator.normal(0.6, 0.2, signature_mask.sum())
        vitals_anomaly[emergency_mask] += emergency_vitals_shift
        table_nonwear_mask = nonwear_mask & (random_generator.random(episode_count) < 0.7)
        motion_energy[table_nonwear_mask] = random_generator.normal(
            -2.7, 0.15, table_nonwear_mask.sum()
        )
        acceleration_peak[table_nonwear_mask] = random_generator.normal(
            -0.3, 0.2, table_nonwear_mask.sum()
        )
        signature_mask = (
            nonwear_mask & ~table_nonwear_mask & (random_generator.random(episode_count) < 0.15)
        )
        acceleration_peak[signature_mask] = random_generator.normal(0.5, 0.3, signature_mask.sum())
        contact_quality[nonwear_mask] = random_generator.normal(0.15, 0.12, nonwear_mask.sum())
        contact_quality[hardware_mask] = random_generator.normal(0.5, 0.25, hardware_mask.sum())
        rssi[bluetooth_mask] -= 4.0
        signature_mask = hardware_mask & (random_generator.random(episode_count) < 0.6)
        battery_slope[signature_mask] *= 2.5
        missing_sensor_mask = random_generator.random(episode_count) < missing_sensor_probability
        for sensor_values in (motion_energy, acceleration_peak, contact_quality, vitals_anomaly):
            sensor_values[missing_sensor_mask] = np.nan
        episode_columns["time"].append(np.full(episode_count, hour))
        episode_columns["device"].append(episode_device_indices)
        episode_columns["cause"].append(episode_causes)
        episode_columns["D_nat"].append(natural_recovery_hours)
        episode_columns["infrastructure_incident_id"].append(
            np.where(
                gateway_outage_mask[gateway_assignments[episode_device_indices]],
                gateway_incident_ids[gateway_assignments[episode_device_indices]],
                -1,
            )
        )
        episode_columns["batt_level"].append(battery_level)
        episode_columns["batt_slope"].append(battery_slope)
        episode_columns["hrs_since_charge"].append(
            hours_since_charge[episode_device_indices].copy()
        )
        episode_columns["motion_E"].append(motion_energy)
        episode_columns["accel_peak"].append(acceleration_peak)
        episode_columns["contact_q"].append(contact_quality)
        episode_columns["rssi_trend"].append(rssi)
        episode_columns["vitals_anom"].append(vitals_anomaly)
        episode_columns["prior_eps"].append(previous_episode_counts[episode_device_indices].copy())
        episode_columns["time_of_day"].append(np.full(episode_count, hour_of_day))
        episode_columns["gateway_id"].append(gateway_assignments[episode_device_indices])
        episode_columns["os_version"].append(os_assignments[episode_device_indices])
        episode_columns["phone_model"].append(phone_model_assignments[episode_device_indices])
        for uniform_column in ("U1", "U2", "U3", "U_hw"):
            episode_columns[uniform_column].append(random_generator.random(episode_count))
        previous_episode_counts[episode_device_indices] += 1
        battery[episode_device_indices[episode_causes == CAUSE_DEAD_BATTERY]] = 100.0
        hours_since_charge[episode_device_indices[episode_causes == CAUSE_DEAD_BATTERY]] = 0.0
    episodes = pd.DataFrame(
        {
            column_name: np.concatenate(column_chunks)
            for column_name, column_chunks in episode_columns.items()
        }
    )
    episodes = episodes[episodes.D_nat > OBSERVATION_DELAY_H].reset_index(drop=True)
    if episodes.empty:
        raise ValueError("The simulated fleet has no observable offline episodes.")
    for key, cohort_assignments in (
        ("gw", episodes["gateway_id"].values),
        ("os", episodes["os_version"].values),
        ("model", episodes["phone_model"].values),
    ):
        group_sizes = cohort_sizes[key]
        hourly_episode_counts = np.zeros((SIMULATION_HOURS, len(group_sizes)))
        np.add.at(hourly_episode_counts, (episodes["time"].values, cohort_assignments), 1)
        cumulative_episode_counts = np.cumsum(hourly_episode_counts, 0)
        rolling_episode_counts = cumulative_episode_counts.copy()
        rolling_episode_counts[3:] -= cumulative_episode_counts[:-3]
        other_devices_per_cohort = np.maximum(group_sizes[cohort_assignments] - 1, 1)
        episodes[f"cohort_{key}"] = (
            hourly_episode_counts[episodes["time"].values, cohort_assignments] - 1
        ) / other_devices_per_cohort
        episodes[f"cohort_{key}_3h"] = (
            rolling_episode_counts[episodes["time"].values, cohort_assignments] - 1
        ) / other_devices_per_cohort
    episodes = episodes.reset_index(drop=True)
    episodes.attrs["gateway_incidents"] = pd.DataFrame(
        incident_records,
        columns=["incident_id", "gateway_id", "onset_hour", "recovery_hour", "registered_devices"],
    )
    episodes.attrs["gateway_observations"] = pd.DataFrame(gateway_observation_records)
    return episodes


def stream_capture_motion(root, maximum_files=None):
    windows = []
    manifest = []
    participant_vectors = []
    cached_ids = set()
    if MOTION_CACHE_WORKBOOK:
        cached = pd.read_excel(MOTION_CACHE_WORKBOOK, sheet_name="motion_windows")
        if not {"Participant", "Minute", "LogStd_g"} <= set(cached.columns):
            raise ValueError("Motion cache requires Participant, Minute and LogStd_g.")
        cached = cached[np.isfinite(pd.to_numeric(cached.LogStd_g, errors="coerce"))].copy()
        if cached.duplicated(["Participant", "Minute"]).any():
            raise ValueError("Duplicate cached motion windows.")
        try:
            old_manifest = pd.read_excel(
                MOTION_CACHE_WORKBOOK, sheet_name="motion_manifest"
            ).to_dict("records")
        except ValueError:
            old_manifest = []
        for participant, group in cached.groupby("Participant"):
            if len(group) < 30:
                raise ValueError(f"Participant {participant}: fewer than 30 valid cached minutes.")
            entry = next((row for row in old_manifest if row["Participant"] == participant), None)
            length = (
                int(entry["RawSamples"] / MOTION_SAMPLE_RATE_HZ / 60)
                if entry
                else int(group.Minute.max()) + 1
            )
            vector = np.full(length, np.nan)
            vector[group.Minute.to_numpy(dtype=int)] = group.LogStd_g.to_numpy(float)
            participant_vectors.append(vector)
            cached_ids.add(str(participant))
            windows.extend(group.to_dict("records"))
            manifest.append(
                entry
                if entry
                else dict(
                    Participant=str(participant),
                    File="cached summary",
                    RawSamples=length * MOTION_SAMPLE_RATE_HZ * 60,
                    ValidMinutes=len(group),
                )
            )
    candidates = [
        path
        for path in Path(root).rglob("P*")
        if path.is_file() and (path.name.endswith(".csv") or path.name.endswith(".csv.gz"))
    ]
    unique = {}
    for path in sorted(candidates):
        participant = path.name.split(".")[0]
        if participant[1:].isdigit():
            unique.setdefault(participant, path)
    all_ids = sorted(set(unique) | cached_ids)
    if maximum_files is not None:
        all_ids = all_ids[:maximum_files]
    if REQUIRE_ALL_CAPTURE_PARTICIPANTS and len(all_ids) < EXPECTED_CAPTURE_PARTICIPANTS:
        raise ValueError(
            f"Found {len(all_ids)} participants including cache; attach all {EXPECTED_CAPTURE_PARTICIPANTS}."
        )
    participants = [participant for participant in all_ids if participant not in cached_ids]
    samples_per_minute = int(MOTION_SAMPLE_RATE_HZ / MOTION_DOWNSAMPLE_FACTOR * 60)
    for participant in participants:
        path = unique[participant]
        names = list(pd.read_csv(path, nrows=0).columns)
        normalized = [str(name).lower().strip() for name in names]
        if not all((axis in normalized for axis in "xyz")):
            raise ValueError(f"{path.name}: missing x/y/z columns")
        columns = [names[normalized.index(axis)] for axis in "xyz"]
        offset = 0
        carry = np.array([], float)
        logs = []
        for chunk in pd.read_csv(path, usecols=columns, dtype=np.float32, chunksize=250000):
            acceleration = chunk.to_numpy()
            start = -offset % MOTION_DOWNSAMPLE_FACTOR
            offset += len(acceleration)
            magnitude = np.sqrt(
                np.square(
                    acceleration[start::MOTION_DOWNSAMPLE_FACTOR].astype(float)
                    * MOTION_INPUT_SCALE_TO_G
                ).sum(axis=1)
            )
            magnitude = np.concatenate([carry, magnitude])
            count = len(magnitude) // samples_per_minute
            if count:
                block = magnitude[: count * samples_per_minute].reshape(count, samples_per_minute)
                valid = np.isfinite(block).all(axis=1)
                batch = np.full(count, np.nan)
                batch[valid] = np.log10(block[valid].std(axis=1) + 1e-06)
                logs.extend(batch.tolist())
            carry = magnitude[count * samples_per_minute :]
        vector = np.asarray(logs)
        if np.isfinite(vector).sum() < 30:
            raise ValueError(f"{participant}: fewer than 30 valid minutes.")
        participant_vectors.append(vector)
        windows.extend(
            (
                dict(Participant=participant, Minute=index, LogStd_g=float(value))
                for index, value in enumerate(vector)
                if np.isfinite(value)
            )
        )
        manifest.append(
            dict(
                Participant=participant,
                File=str(path),
                RawSamples=offset,
                ValidMinutes=int(np.isfinite(vector).sum()),
            )
        )
        if len(manifest) % 10 == 0:
            export_results_workbook(
                {"motion_windows": windows, "motion_manifest": manifest},
                {
                    "status": "motion_extraction_checkpoint",
                    "completed_participants": len(manifest),
                    "motion_sample_rate": MOTION_SAMPLE_RATE_HZ,
                    "downsample_factor": MOTION_DOWNSAMPLE_FACTOR,
                    "acceleration_scale_to_g": MOTION_INPUT_SCALE_TO_G,
                },
                Path(RESULTS_DIRECTORY, "study_results.xlsx"),
            )
        LOGGER.info(
            "Streamed motion participant %s (%d/%d)", participant, len(manifest), len(all_ids)
        )
    if not windows:
        return None
    rates = {}
    for duration in [1, 5, 15, 30, 60]:
        values = []
        for vector in participant_vectors:
            if len(vector) < duration:
                continue
            complete = (
                np.convolve(np.isfinite(vector).astype(int), np.ones(duration, int), "valid")
                == duration
            )
            still = (
                np.convolve((vector < np.log10(0.005)).astype(int), np.ones(duration, int), "valid")
                == duration
            )
            if complete.any():
                values.append(float(still[complete].mean()))
        rates[duration] = float(np.mean(values)) if values else np.nan
    return dict(
        log_std=np.array([row["LogStd_g"] for row in windows]),
        windows=windows,
        manifest=manifest,
        input_files=sorted((row["Participant"] for row in manifest)),
        n_files=len(manifest),
        n_windows=len(windows),
        fa_curve=pd.Series(rates, name="FalseNonwearAlarmRate"),
    )


def ticket_membership(context):
    episodes = context["test"]
    decision_time = episodes.time.to_numpy() + OBSERVATION_DELAY_H
    membership = np.zeros(len(episodes), dtype=bool)
    for ticket in context["tickets"].itertuples():
        before_close = (
            decision_time <= ticket.closed_hour
            if ticket.right_censored
            else decision_time < ticket.closed_hour
        )
        membership |= (
            (episodes.gateway_id.to_numpy() == ticket.gateway_id)
            & (decision_time >= ticket.detected_hour)
            & before_close
        )
    return membership


def tolerance_emergency_threshold(positive_scores, miss_target, failure_probability=0.05):
    """One-sided order-statistic tolerance threshold for positive-class scores.

    For an independently fitted score function and iid calibration/test
    positives, a rank j is eligible when BinomialCDF(j-1;n,epsilon)<=delta.
    Ties are conservative for strict score<threshold misses. No lower clip is
    applied. With insufficient positives the threshold is zero: every episode
    is escalated. This exposes sample scarcity rather than claiming that the
    heuristic 0.05 fallback certifies recall. Temporal/deployment shift can
    invalidate the iid interpretation; empirical violation rates are exported.
    This is established tolerance-bound methodology, not a new guarantee.
    """
    scores = np.sort(np.asarray(positive_scores, dtype=float))
    if not 0 < miss_target < 1 or not 0 < failure_probability < 1:
        raise ValueError("Risk and confidence parameters must be in (0, 1).")
    ranks = np.arange(1, len(scores) + 1)
    eligible = ranks[stats.binom.cdf(ranks - 1, len(scores), miss_target) <= failure_probability]
    if not len(eligible):
        return (0.0, "all_escalate_insufficient_positive_support")
    return (float(scores[int(eligible[-1]) - 1]), "iid_positive_class_tolerance_bound")


def tune_common_rule(episodes, scores, target, outcomes, exposure):
    positives = episodes.cause.eq(CAUSE_CLINICAL_EMERGENCY).to_numpy()
    threshold = (
        float(np.quantile(scores[positives], target, method="lower")) if positives.any() else 0.0
    )
    candidates = []
    for parameters, initial in rule_candidates(episodes, missing_aware=True):
        selected = initial.copy()
        selected[scores >= threshold] = ACTION_NURSE
        metrics = compute_policy_metrics(
            selected, outcomes, episodes, exposure, DEFAULT_LOSS_PARAMETERS
        )
        candidates.append(
            ((metrics["Mean_Loss"], metrics["NurseFirst_nonEm_per_1000dd"]), parameters)
        )
    return min(candidates, key=lambda candidate: candidate[0])[1]


In [ ]:
def workbook_value(value):
    if isinstance(value, np.generic):
        value = value.item()
    if value is None:
        return None
    if isinstance(value, float) and (not np.isfinite(value)):
        return None if np.isnan(value) else str(value)
    if isinstance(value, (dict, list, tuple, np.ndarray)):
        return json.dumps(value, default=serializable_metadata, sort_keys=True)
    return value


PAPER_REVISION_SOURCE_SHA256 = "f420d24228a1e630c818dd3257bc00a7899ca7140e034656343b62b898df8c08"


def simulate_fleet(seed, **parameters):
    parameters.setdefault("device_count", N_DEVICES)
    return simulate_fleet_reference(seed, **parameters)


def paper_history(seed, role, blocks):
    frames = []
    for block in range(blocks):
        frame = simulate_fleet(seed * 100 + role * 10 + block, motion_reference=motion_reference)
        frame.attrs.clear()
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def paper_score(head, frame, override=False):
    score_vector = head["model"].predict_proba(frame[head["features"]])[:, 1]
    if override:
        override_mask = (
            (frame.vitals_anom.to_numpy() > 2.0)
            | (frame.accel_peak.to_numpy() > 0.45)
            | frame.vitals_anom.isna().to_numpy()
            | frame.accel_peak.isna().to_numpy()
        )
        score_vector[override_mask] = 1.0
    return score_vector


def paper_gate(cal, score, target):
    return tolerance_emergency_threshold(
        score[cal.cause.eq(CAUSE_CLINICAL_EMERGENCY).to_numpy()], target, 0.05
    )[0]


In [ ]:
def observable_suspicion(test, score, threshold):
    return (
        (score >= threshold)
        | (test.vitals_anom.to_numpy() > 2.0)
        | (test.accel_peak.to_numpy() > 0.45)
        | test.vitals_anom.isna().to_numpy()
        | test.accel_peak.isna().to_numpy()
    )


def public_log_anchor():
    import urllib.request, hashlib

    url = "https://raw.githubusercontent.com/logpai/loghub/master/BGL/BGL_2k.log_structured.csv"
    path = (
        Path(PUBLIC_LOG_CSV)
        if PUBLIC_LOG_CSV
        else Path(RESULTS_DIRECTORY, "BGL_2k.log_structured.csv")
    )
    if not path.exists():
        with urllib.request.urlopen(url, timeout=60) as response:
            source_bytes = response.read()
        path.write_bytes(source_bytes)
    frame = pd.read_csv(path)
    required = {"Timestamp", "Node", "Label"}
    if not required.issubset(frame):
        raise ValueError("BGL anchor requires Timestamp, Node, Label.")
    frame["Window"] = pd.to_numeric(frame.Timestamp, errors="raise") // 3600
    frame["Alarm"] = frame.Label.ne("-")
    alarms = frame[frame.Alarm]
    rows = []
    for window, window_records in alarms.groupby("Window"):
        rows.append(
            dict(
                Window=int(window),
                AlarmRows=len(window_records),
                DistinctNodes=int(window_records.Node.nunique()),
                SameNodeWindowGroups=int(window_records.groupby("Node").ngroups),
                Scope="Observed labeled system alarms; shared root cause and WBAN transfer not established",
            )
        )
    return (
        rows,
        [
            dict(
                Source=url,
                SHA256=hashlib.sha256(path.read_bytes()).hexdigest(),
                Rows=len(frame),
                AlarmRows=len(alarms),
                Scope="Public 2,000-line BGL sample; alarm co-occurrence anchor only, not gateway outage ground truth",
            )
        ],
    )


def revision_scenarios():
    scenarios = {
        "Base": {},
        "MissingSensors": dict(missing_sensor_probability=0.3),
        "WeakSignals": dict(emergency_fall_probability=0.3, emergency_vitals_shift=1.0),
        "MixedOutages": dict(
            gateway_outage_probability=0.01,
            outage_emergency_probability=0.02,
            test_emergency_multiplier=3.0,
        ),
    }
    return scenarios


def robustness_scenarios():
    return {
        f"Outage{p}_Participation{c}_Gateways{g}": dict(
            gateway_outage_probability=p, gateway_failure_participation=c, gateway_count=g
        )
        for p in [0.0005, 0.002, 0.01]
        for c in [0.25, 0.5, 1.0]
        for g in [2, 5, 10]
    }


def run_paper_revision():
    directory = Path(RESULTS_DIRECTORY)
    directory.mkdir(parents=True, exist_ok=True)
    tables = {
        k: []
        for k in [
            "results",
            "dispatch",
            "calibration",
            "sample_sizes",
            "suppression_audit",
            "paired_effects",
            "robustness_regions",
            "precision",
            "public_anchor",
            "provenance",
        ]
    }
    protocol = dict(
        status="running",
        source_hash=PAPER_REVISION_SOURCE_SHA256,
        origin="Independent simulation evaluation of context-aware WBAN fault management",
        independent_seeds="New realizations; previous PDF tables are historical comparisons only",
        main_endpoint="Base full-minus-no-cohort non-emergency nurse-first workload at target .10 without override",
        safety="Report paired miss difference and CI; no noninferiority claim without justified clinical margin",
        suppression="Guard preserves original action on observed suspicion; cannot protect undetected emergencies",
        robustness="Frozen Base-trained models; 27 outage-frequency x gateway-participation x gateway-count cells",
        public_anchor="BGL public sample establishes observed alarm co-occurrence, not shared outage causality",
        queue_scope="Exogenous-arrival service replay; separate from onset-to-operational-resolution",
        calibration="Single positive-class order-statistic threshold, delta .05; iid and deployment assumptions conditional",
    )
    if RUN_PUBLIC_LOG_ANCHOR:
        tables["public_anchor"], tables["provenance"] = public_log_anchor()
    jobs = [
        ("Main", realization_seed, revision_scenarios())
        for realization_seed in MAIN_REALIZATION_SEEDS
    ]
    jobs += [
        ("Precision", realization_seed, revision_scenarios())
        for realization_seed in PRECISION_REALIZATION_SEEDS
    ]
    jobs += [
        ("Robustness", realization_seed, robustness_scenarios())
        for realization_seed in ROBUSTNESS_REALIZATION_SEEDS
    ]
    for study, seed, scenarios in jobs:
        training_episodes = paper_history(seed, 0, 3)
        probability_calibration = paper_history(seed, 1, 1)
        tuning = paper_history(seed, 2, 1)
        threshold_calibration = paper_history(seed, 3, 8)
        fitted = fit_posteriors(
            training_episodes, probability_calibration, tuning, tuning, seed=seed
        )
        matrix = estimate_loss_matrix(
            training_episodes, compute_action_outcomes(training_episodes)["loss"]
        )
        excluded = set(ABLATION_FEATURE_GROUPS["no_cohort"])
        heads = {
            "Full": fit_emergency_model(training_episodes, FEATURE_COLUMNS, seed),
            "NoCohort": fit_emergency_model(
                training_episodes, [c for c in FEATURE_COLUMNS if c not in excluded], seed
            ),
        }
        for scenario, parameters in scenarios.items():
            fleet = simulate_fleet(seed, motion_reference=motion_reference, **parameters)
            test = fleet[fleet.time >= 0.75 * SIMULATION_HOURS].reset_index(drop=True)
            context = incident_context(fleet, test, seed)
            posterior = predict_posteriors(fitted, test)
            positive = test.cause.eq(CAUSE_CLINICAL_EMERGENCY).to_numpy()
            tables["sample_sizes"].append(
                dict(
                    Study=study,
                    Seed=seed,
                    Scenario=scenario,
                    Episodes=len(test),
                    Emergencies=int(positive.sum()),
                    CalibrationEmergencies=int(
                        threshold_calibration.cause.eq(CAUSE_CLINICAL_EMERGENCY).sum()
                    ),
                    **parameters,
                )
            )
            targets = PAPER_TARGETS if study == "Main" else [0.1]
            overrides = [False, True] if study != "Robustness" else [False]
            for head, model in heads.items():
                for override in overrides:
                    calibration_scores = paper_score(model, threshold_calibration, override)
                    score = paper_score(model, test, override)
                    tuning_scores = paper_score(model, tuning, override)
                    for target in targets:
                        threshold = paper_gate(threshold_calibration, calibration_scores, target)
                        rule = tune_common_rule(
                            tuning,
                            tuning_scores,
                            target,
                            compute_action_outcomes(tuning),
                            N_DEVICES * SIMULATION_DAYS,
                        )
                        raw = initial_operational_actions(test, posterior, matrix, rule)
                        raw["GatewayCorrelation"] = np.where(
                            test.cohort_gw.to_numpy() > 0.5, ACTION_WAIT, ACTION_RECONNECT
                        )
                        tables["calibration"].append(
                            dict(
                                Study=study,
                                Seed=seed,
                                Scenario=scenario,
                                Detector=head,
                                Override=override,
                                Target=target,
                                Threshold=threshold,
                            )
                        )
                        for policy in [
                            "CausePolicy",
                            "TunedRule",
                            "GatewayCorrelation",
                            "AlwaysNurse",
                        ]:
                            actions = raw[policy].copy()
                            actions[score >= threshold] = ACTION_NURSE
                            for mode in ["PerDevice", "NaiveShared", "GuardedShared"]:
                                current = dict(context)
                                protected = (
                                    observable_suspicion(test, score, threshold)
                                    if mode == "GuardedShared"
                                    else np.zeros(len(test), bool)
                                )
                                current["suppression_protected"] = protected
                                evaluated = evaluate_incident_policy(
                                    current,
                                    actions,
                                    "PerDevice" if mode == "PerDevice" else "NaiveShared",
                                )
                                tags = dict(
                                    Study=study,
                                    Seed=seed,
                                    Scenario=scenario,
                                    Detector=head,
                                    Override=override,
                                    Target=target,
                                    Policy=policy,
                                    Infrastructure=mode,
                                )
                                tables["results"].append(dict(**tags, **evaluated["metrics"]))
                                suppress = (
                                    ticket_membership(current)
                                    & (actions != ACTION_NURSE)
                                    & (actions != ACTION_WAIT)
                                    & ~protected
                                    if mode != "PerDevice"
                                    else np.zeros(len(test), bool)
                                )
                                assert not np.any(suppress & protected)
                                tables["suppression_audit"].append(
                                    dict(
                                        **tags,
                                        Suppressed=int(suppress.sum()),
                                        EmergencySuppressed=int((suppress & positive).sum()),
                                        SuspectedSuppressed=int(
                                            (
                                                suppress
                                                & observable_suspicion(test, score, threshold)
                                            ).sum()
                                        ),
                                        Protected=int(protected.sum()),
                                    )
                                )
                                if (
                                    study != "Robustness"
                                    and head == "Full"
                                    and override
                                    and (target == 0.1)
                                ):
                                    for capacity in [5, 10, None]:
                                        tables["dispatch"].append(
                                            dict(
                                                **tags,
                                                **closed_loop_dispatch(
                                                    current,
                                                    evaluated,
                                                    score,
                                                    capacity,
                                                    "risk_deadline",
                                                ),
                                            )
                                        )
        protocol["last_completed_job"] = f"{study}:{seed}"
        export_results_workbook(tables, protocol, directory / "paper_revision_results.xlsx")
        LOGGER.info("Completed %s seed %s", study, seed)
    frame = pd.DataFrame(tables["results"])
    keys = ["Study", "Scenario", "Override", "Target", "Policy", "Infrastructure"]
    for group, comparison_group in frame.groupby(keys):
        candidate_by_seed = comparison_group[comparison_group.Detector == "Full"].set_index("Seed")
        baseline_by_seed = comparison_group[comparison_group.Detector == "NoCohort"].set_index(
            "Seed"
        )
        for metric in ["NurseFirst_nonEm_per_1000dd", "Missed_Emerg", "Mean_Loss", "Emerg_TTR"]:
            difference = (candidate_by_seed[metric] - baseline_by_seed[metric]).dropna()
            low, high = bootstrap_interval(difference.to_numpy())
            row = dict(zip(keys, group))
            row.update(
                Metric=metric,
                Difference=float(difference.mean()),
                Lower95=low,
                Upper95=high,
                Seeds=len(difference),
                Interpretation="Pointwise exploratory paired bootstrap; not equivalence",
            )
            tables["paired_effects"].append(row)
            if group[0] == "Robustness" and metric == "NurseFirst_nonEm_per_1000dd":
                tables["robustness_regions"].append(
                    dict(
                        **row,
                        BenefitSupported=bool(high < 0),
                        BenefitDisappearsOrUncertain=bool(high >= 0),
                    )
                )
    tables["baseline_and_guard_effects"] = []
    for contrast, column, first, second in [
        ("Cause minus gateway correlation", "Policy", "CausePolicy", "GatewayCorrelation"),
        ("Cause minus tuned rule", "Policy", "CausePolicy", "TunedRule"),
        ("Guarded minus unguarded shared", "Infrastructure", "GuardedShared", "NaiveShared"),
    ]:
        grouping = [
            c
            for c in [
                "Study",
                "Scenario",
                "Detector",
                "Override",
                "Target",
                "Policy",
                "Infrastructure",
            ]
            if c != column
        ]
        for group, comparison_group in frame.groupby(grouping):
            candidate_by_seed = comparison_group[comparison_group[column] == first].set_index(
                "Seed"
            )
            baseline_by_seed = comparison_group[comparison_group[column] == second].set_index(
                "Seed"
            )
            for metric in [
                "Mean_Loss",
                "Missed_Emerg",
                "NurseFirst_nonEm_per_1000dd",
                "Emerg_TTR",
                "EmergencyActionsSuppressed",
            ]:
                paired_differences = (candidate_by_seed[metric] - baseline_by_seed[metric]).dropna()
                if not len(paired_differences):
                    continue
                low, high = bootstrap_interval(paired_differences.to_numpy())
                row = dict(zip(grouping, group))
                row.update(
                    Contrast=contrast,
                    Metric=metric,
                    Difference=float(paired_differences.mean()),
                    Lower95=low,
                    Upper95=high,
                    Seeds=len(paired_differences),
                    IntervalScope="Pointwise exploratory",
                )
                tables["baseline_and_guard_effects"].append(row)
    sizes = pd.DataFrame(tables["sample_sizes"])
    for (study, scenario), comparison_group in sizes.groupby(["Study", "Scenario"]):
        tables["precision"].append(
            dict(
                Study=study,
                Scenario=scenario,
                Realizations=len(comparison_group),
                PooledEmergencies=int(comparison_group.Emergencies.sum()),
                MeanEmergenciesPerRealization=float(comparison_group.Emergencies.mean()),
                Interpretation="More independent runs improve precision; sample count alone does not prove power for a clinical margin",
            )
        )
    add_management_reporting(tables)
    protocol["status"] = "complete"
    export_results_workbook(tables, protocol, directory / "paper_revision_results.xlsx")
    main = frame[
        (frame.Study == "Main")
        & (frame.Policy == "CausePolicy")
        & (frame.Infrastructure == "PerDevice")
        & ~frame.Override
    ]
    for scenario, comparison_group in main.groupby("Scenario"):
        fig, ax = plt.subplots(figsize=(6, 4))
        for head, h in comparison_group.groupby("Detector"):
            q = h.groupby("Target")[["Missed_Emerg", "NurseFirst_nonEm_per_1000dd"]].mean()
            ax.plot(q.NurseFirst_nonEm_per_1000dd, q.Missed_Emerg, marker="o", label=head)
        ax.axhline(0.1, ls="--", color="black")
        ax.set(
            xlabel="Non-emergency nurse-first contacts / 1,000 device-days",
            ylabel="Miss rate",
            title=scenario,
        )
        ax.legend()
        fig.tight_layout()
        fig.savefig(directory / f"frontier_{scenario}.pdf", dpi=1200)
        plt.close(fig)
    return tables


def add_management_reporting(tables):
    tables["contributions"] = [
        dict(
            Number=1,
            Contribution="Coordinated management architecture separating emergency escalation, cause-based expected-loss decisions, and shared repair",
        ),
        dict(
            Number=2,
            Contribution="Observable suspicion-based guard preserving selected actions on flagged episodes; residual emergency suppression measured",
        ),
        dict(
            Number=3,
            Contribution="Reproducible paired evaluation of workload, safety, and suppression across deployment shifts and 27 robustness settings",
        ),
    ]
    tables["claim_scope"] = [
        dict(
            Claim="Cohort workload savings",
            Comparator="Same cause policy with versus without cohort inputs in emergency detector; not per-device rule versus proposed",
        ),
        dict(
            Claim="Safety",
            Comparator="Observed rates and paired intervals; no equivalence or noninferiority established",
        ),
        dict(
            Claim="Robustness",
            Comparator="Workload evidence across tested cells; safety robustness is assessed separately",
        ),
        dict(
            Claim="Guard",
            Comparator="Prevents suppression of flagged actions; does not prevent score masking or identify every emergency",
        ),
        dict(
            Claim="External anchor",
            Comparator="BGL sample describes cross-domain alarm co-occurrence only",
        ),
        dict(
            Claim="Availability",
            Comparator="Notebook records reproducibility information; public code release must be verified separately",
        ),
    ]
    frame = pd.DataFrame(tables["results"])
    keys = ["Study", "Scenario", "Detector", "Override", "Target", "Policy", "Infrastructure"]
    metrics = [
        "Mean_Loss",
        "Cost_Action",
        "Cost_Data",
        "Cost_Clinical",
        "Cost_Infrastructure",
        "Missed_Emerg",
        "NurseFirst_nonEm_per_1000dd",
    ]
    loss = frame.groupby(keys, dropna=False)[metrics].mean().reset_index()
    loss["ReconstructedLoss"] = loss[
        ["Cost_Action", "Cost_Data", "Cost_Clinical", "Cost_Infrastructure"]
    ].sum(axis=1)
    loss["ReconciliationError"] = loss.Mean_Loss - loss.ReconstructedLoss
    if not np.allclose(loss.ReconciliationError, 0, atol=1e-07):
        raise ValueError("Loss components do not reconcile.")
    tables["loss_decomposition"] = loss.to_dict("records")
    guard = pd.DataFrame(tables["suppression_audit"])
    protected = guard[guard.Infrastructure == "GuardedShared"]
    if protected.SuspectedSuppressed.sum() != 0:
        raise ValueError("Guard violated its observable suspicion property.")
    tables["guard_property"] = [
        dict(
            Property="No action suppression among flagged episodes",
            Violations=int(protected.SuspectedSuppressed.sum()),
            ResidualEmergencySuppressions=int(protected.EmergencySuppressed.sum()),
            Scope="Residual total spans experiment configurations; do not treat repeated configurations as unique emergencies",
        )
    ]
    sweep = loss[(loss.Study == "Main") & (loss.Infrastructure == "PerDevice")]
    tables["policy_operating_points"] = sweep.to_dict("records")
    comparisons = []
    for (scenario, head, override), group in sweep.groupby(["Scenario", "Detector", "Override"]):
        candidate = group[group.Policy == "CausePolicy"]
        for _, point in candidate.iterrows():
            for policy in ["TunedRule", "GatewayCorrelation"]:
                reference = group[group.Policy == policy]
                if reference.empty:
                    continue
                nearest = reference.iloc[
                    (reference.Missed_Emerg - point.Missed_Emerg).abs().argmin()
                ]
                gap = float(nearest.Missed_Emerg - point.Missed_Emerg)
                comparisons.append(
                    dict(
                        Scenario=scenario,
                        Detector=head,
                        Override=bool(override),
                        Policy=policy,
                        CauseTarget=float(point.Target),
                        BaselineTarget=float(nearest.Target),
                        CauseMiss=float(point.Missed_Emerg),
                        BaselineMiss=float(nearest.Missed_Emerg),
                        MissGap=gap,
                        WithinDeclaredTolerance=abs(gap) <= OBSERVED_MISS_MATCH_TOLERANCE,
                        CauseWorkload=float(point.NurseFirst_nonEm_per_1000dd),
                        BaselineWorkload=float(nearest.NurseFirst_nonEm_per_1000dd),
                        Interpretation="Nearest measured operating points; descriptive post hoc pairing, not safety equivalence or a new evaluated threshold",
                    )
                )
    tables["nearest_measured_points"] = comparisons
    directory = Path(RESULTS_DIRECTORY)
    directory.mkdir(parents=True, exist_ok=True)
    for scenario, group in sweep[(sweep.Detector == "Full") & ~sweep.Override.astype(bool)].groupby(
        "Scenario"
    ):
        fig, ax = plt.subplots(figsize=(6, 4))
        for policy in ["CausePolicy", "TunedRule", "GatewayCorrelation"]:
            policy_records = group[group.Policy == policy].sort_values("Target")
            ax.plot(
                policy_records.NurseFirst_nonEm_per_1000dd,
                policy_records.Missed_Emerg,
                marker="o",
                label=policy,
            )
            for _, operating_point in policy_records.iterrows():
                ax.annotate(
                    f"{operating_point.Target:.2f}",
                    (operating_point.NurseFirst_nonEm_per_1000dd, operating_point.Missed_Emerg),
                    fontsize=7,
                )
        ax.set(
            xlabel="Non-emergency nurse-first contacts / 1,000 device-days",
            ylabel="Observed miss rate",
            title=scenario,
        )
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(directory / f"policy_tradeoff_{scenario}.pdf", dpi=1200)
        plt.close(fig)
    base = loss[
        (loss.Study == "Main")
        & (loss.Scenario == "Base")
        & (loss.Detector == "Full")
        & ~loss.Override.astype(bool)
        & (loss.Target == 0.1)
        & (loss.Infrastructure == "PerDevice")
    ]
    base = base[base.Policy.isin(["CausePolicy", "TunedRule", "GatewayCorrelation"])].set_index(
        "Policy"
    )
    fig, ax = plt.subplots(figsize=(6, 4))
    bottom = np.zeros(len(base))
    for metric in ["Cost_Action", "Cost_Data", "Cost_Clinical", "Cost_Infrastructure"]:
        values = base[metric].to_numpy()
        ax.bar(base.index, values, bottom=bottom, label=metric.replace("Cost_", ""))
        bottom += values
    ax.set(
        ylabel="Mean modeled loss per episode",
        title="Base loss decomposition: 20 main realizations",
    )
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(directory / "baseline_loss_decomposition.pdf", dpi=1200)
    plt.close(fig)


In [ ]:
Path(RESULTS_DIRECTORY).mkdir(parents=True, exist_ok=True)

if RESULTS_INPUT_WORKBOOK:
    source = Path(RESULTS_INPUT_WORKBOOK)
    if not source.exists():
        raise FileNotFoundError(source)
    with pd.ExcelFile(source) as book:
        paper_results = {
            name: pd.read_excel(book, sheet_name=name).replace({np.nan: None}).to_dict("records")
            for name in book.sheet_names
            if name not in ["Overview", "Protocol", "TableIndex"]
        }
        previous = pd.read_excel(book, sheet_name="Protocol")
        previous_protocol = dict(zip(previous.iloc[:, 0], previous.iloc[:, 1]))
    add_management_reporting(paper_results)
    previous_protocol.update(
        status="complete_reporting_revision",
        reporting_source_hash=PAPER_REVISION_SOURCE_SHA256,
        data_origin=str(source),
        scope="Existing completed experiment results; no new simulations",
    )
    destination = Path(RESULTS_DIRECTORY, "paper_reporting_results.xlsx")
    if destination.resolve() == source.resolve():
        raise ValueError("Choose a separate output path to preserve the source workbook.")
    export_results_workbook(paper_results, previous_protocol, destination)
else:
    real_motion_profile = stream_capture_motion(CAPTURE24_DIRECTORY, CAPTURE_MAXIMUM_FILES)
    if REQUIRE_REAL_MOTION and real_motion_profile is None:
        raise ValueError("Attach Capture-24 or the complete minute-summary workbook.")
    motion_reference = np.asarray(real_motion_profile["log_std"]) if real_motion_profile else None
    paper_results = run_paper_revision()
